# EcoSort Open-Source Auto Dataset + Training

This notebook builds an EcoSort dataset from legal open-source/public datasets, adds your own Drive photos if present, trains a MobileNetV2 classifier, and exports a TensorFlow.js model for the web app.

Sources used by the notebook:
- TrashNet: glass, paper, cardboard, plastic, metal, trash.
- TACO: litter images with COCO annotations.

Important: Internet/open datasets help, but for 90% real demo accuracy you should still add webcam-style photos from your actual dustbin setup into `MyDrive/EcoSortAI_Project/EcoSortDataset_UserPhotos`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil, random, pathlib, subprocess, sys
from pathlib import Path
from PIL import Image

PROJECT_DIR = Path('/content/drive/MyDrive/EcoSortAI_Project')
DATASET_DIR = PROJECT_DIR / 'EcoSortDataset_OpenSource'
USER_PHOTOS_DIR = PROJECT_DIR / 'EcoSortDataset_UserPhotos'
EXPORT_DIR = PROJECT_DIR / 'EcoSortModel_tfjs'

CLASSES = ['Paper','Cardboard','Plastic Bottle','Glass Bottle','Metal Can','Food Waste','Banana Peel','Apple','Chip Packet','Mixed Covered Bag']
for cls in CLASSES:
    (DATASET_DIR / cls).mkdir(parents=True, exist_ok=True)
    (USER_PHOTOS_DIR / cls).mkdir(parents=True, exist_ok=True)

print('Dataset folder:', DATASET_DIR)
print('Optional user photos folder:', USER_PHOTOS_DIR)

In [ ]:
!pip -q install datasets tensorflowjs
from datasets import load_dataset

def safe_save_image(img, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        if img.mode != 'RGB':
            img = img.convert('RGB')
        img.save(dst, quality=92)
        return True
    except Exception as exc:
        print('skip image:', exc)
        return False

TRASHNET_MAP = {
    'paper': 'Paper',
    'cardboard': 'Cardboard',
    'plastic': 'Plastic Bottle',
    'glass': 'Glass Bottle',
    'metal': 'Metal Can',
    'trash': 'Chip Packet'
}

print('Downloading TrashNet from Hugging Face...')
trashnet = load_dataset('garythung/trashnet')
split_name = list(trashnet.keys())[0]
ds = trashnet[split_name]
names = ds.features['label'].names
count = 0
for i, row in enumerate(ds):
    src_label = names[row['label']].lower().strip()
    target = TRASHNET_MAP.get(src_label)
    if not target:
        continue
    if safe_save_image(row['image'], DATASET_DIR / target / f'trashnet_{src_label}_{i:05d}.jpg'):
        count += 1
print('TrashNet images added:', count)

In [ ]:
print('Downloading TACO toolkit and images...')
%cd /content
if not Path('/content/TACO').exists():
    !git clone --depth 1 https://github.com/pedropro/TACO.git
%cd /content/TACO
!python download.py

ann_path = Path('/content/TACO/data/annotations.json')
with ann_path.open('r', encoding='utf-8') as f:
    taco = json.load(f)

images = {img['id']: img for img in taco['images']}
categories = {cat['id']: cat['name'] for cat in taco['categories']}

def taco_target(category_name):
    n = category_name.lower()
    if 'paper' in n or 'newspaper' in n or 'magazine' in n:
        return 'Paper'
    if 'carton' in n or 'cardboard' in n:
        return 'Cardboard'
    if 'glass' in n or 'bottle' in n and 'plastic' not in n:
        return 'Glass Bottle'
    if 'plastic bottle' in n or ('bottle' in n and 'glass' not in n):
        return 'Plastic Bottle'
    if 'can' in n or 'metal' in n or 'aluminium' in n or 'aluminum' in n:
        return 'Metal Can'
    if 'food' in n:
        return 'Food Waste'
    if 'plastic bag' in n or 'wrapper' in n or 'packet' in n or 'snack' in n:
        return 'Chip Packet'
    return None

added = 0
for idx, ann in enumerate(taco['annotations']):
    target = taco_target(categories.get(ann['category_id'], ''))
    if not target:
        continue
    img_info = images[ann['image_id']]
    img_path = Path('/content/TACO/data') / img_info['file_name']
    if not img_path.exists():
        continue
    try:
        with Image.open(img_path).convert('RGB') as img:
            x, y, w, h = ann['bbox']
            pad = 16
            left = max(0, int(x) - pad)
            top = max(0, int(y) - pad)
            right = min(img.width, int(x + w) + pad)
            bottom = min(img.height, int(y + h) + pad)
            if right - left < 40 or bottom - top < 40:
                continue
            crop = img.crop((left, top, right, bottom))
            if safe_save_image(crop, DATASET_DIR / target / f'taco_{idx:05d}.jpg'):
                added += 1
    except Exception as exc:
        print('skip taco crop:', exc)

print('TACO crops added:', added)

In [ ]:
print('Merging optional user photos...')
merged = 0
for cls in CLASSES:
    src_dir = USER_PHOTOS_DIR / cls
    if not src_dir.exists():
        continue
    for src in src_dir.glob('*'):
        if src.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.webp']:
            continue
        dst = DATASET_DIR / cls / f'user_{src.stem}{src.suffix.lower()}'
        if not dst.exists():
            shutil.copy2(src, dst)
            merged += 1
print('User photos merged:', merged)

print('\nFinal class counts:')
too_low = []
for cls in CLASSES:
    n = len([p for p in (DATASET_DIR / cls).glob('*') if p.suffix.lower() in ['.jpg','.jpeg','.png','.webp']])
    print(f'{cls}: {n}')
    if n < 80:
        too_low.append(cls)

if too_low:
    print('\nNeed more real photos for:', too_low)
    print('Training can run, but 90% real-world accuracy is not realistic for low-count classes.')

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=CLASSES
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=CLASSES
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomContrast(0.18),
])

base = tf.keras.applications.MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(len(CLASSES), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3, monitor='val_loss')
]
history = model.fit(train_ds, validation_data=val_ds, epochs=16, callbacks=callbacks)

In [ ]:
print('Fine-tuning last MobileNetV2 layers...')
base.trainable = True
for layer in base.layers[:-35]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)
loss, acc = model.evaluate(val_ds)
print('Validation accuracy:', round(acc * 100, 2), '%')

In [ ]:
import tensorflowjs as tfjs
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
tfjs.converters.save_keras_model(model, str(EXPORT_DIR))
metadata = {
    'labels': CLASSES,
    'imageSize': 224,
    'preprocess': 'embedded_mobilenet_v2',
    'source': 'TrashNet + TACO + optional user photos',
    'validationAccuracy': float(acc)
}
with open(EXPORT_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Exported model:', EXPORT_DIR)
print('Copy these files into web app custom-model folder: model.json, metadata.json, group*.bin')